# HHL: preparing a linear-system solution state

PHYS 3810, session 3. After the five-minute QPE refresher, this notebook supplies a **20-minute core** and a **15-minute optional expansion**. Timing is provisional pending classroom rehearsal. The shared session also reserves 20 minutes for Shor and five minutes for limitations.

**Before class:** run all cells to check the environment. **In class:** restart and clear outputs, make the predictions below, then run the relevant cells. Validation executes every code cell, including reference checks.

Our five-qubit demonstration prepares a state proportional to $A^{-1}b$, conditioned on a success flag. It does not return a classical solution vector. We use a finite 2x2 positive Hermitian matrix with eigenvalues in **{1,2,3,4,5,6,7}** and a normalized complex length-two vector $b$. This exact-spectrum example is deliberately restricted; it is not a general linear-system solver or a quantum-speedup demonstration.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "phys3810" / "hhl.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Keep the extracted course folder together and launch Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Core, minutes 0-5: predict the register contents

Fix three counting qubits and $U=\exp(+iA\pi/4)$. An eigenvalue $\lambda$ has phase $\lambda/8$, so the measured counting integer would be **$y=\lambda$**, not $\lambda/8$. Labels 1 through 7 are all positive: the high bit is not a sign bit. Zero and the alias at 8 are excluded.

Qubits 0-2 are `count`, qubit 3 is `system`, and qubit 4 is `success`. A statevector is ordered **success tensor system tensor count**. Start with all five qubits zero. The source prepares $b$, applies canonical QPE, rotates success by $2\arcsin(1/y)$ conditional on the complete counting label, then applies the **inverse of that same QPE block**. It does not undo preparation of $b$. At label zero the reciprocal rotation is identity.

Before running the parameter cell, predict the counting labels for
$A_0=\begin{pmatrix}3/2&1/2\\1/2&3/2\end{pmatrix}$, whose eigenvalues are 1 and 2. Which eigencomponent receives the larger success amplitude? Should count return to zero on the failure branch too? Will the success flag itself return to zero?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import ClassicalRegister
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from src.phys3810.hhl import hhl_circuit

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."
print("Python:", sys.version.split()[0], "| Qiskit:", version("qiskit"))
print("Interpreter:", sys.executable)
ATOL = 1e-10
PAULIS = {
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.diag([1, -1]),
}


def measurement_copy(circuit, basis):
    """Measure system and flag from the same shot; leave the source unmeasured."""
    sampled = circuit.copy()
    if basis == "X":
        sampled.h(3)
    elif basis == "Y":
        sampled.sdg(3)
        sampled.h(3)
    elif basis != "Z":
        raise ValueError("Choose X, Y, or Z.")
    system = ClassicalRegister(1, "system_readout")
    success = ClassicalRegister(1, "success_readout")
    sampled.add_register(system, success)
    sampled.measure(3, system[0])
    sampled.measure(4, success[0])
    return sampled


def selected_system_bits(system_bits, success_bits):
    """Preserve shot alignment. Empty selection is data, not an invented mean."""
    if len(system_bits) != len(success_bits):
        raise ValueError("System and success samples must have matching shot counts.")
    selected = [int(bit) for bit, flag in zip(system_bits, success_bits, strict=True) if flag == "1"]
    return np.array(selected, dtype=int)

In [ ]:
# Editable parameters: use the supported exact-spectrum family and normalized b.
matrix_entries = [[1.5, 0.5], [0.5, 1.5]]
rhs_amplitudes = [1, 0]
A = np.array(matrix_entries, dtype=complex)
b = np.array(rhs_amplitudes, dtype=complex)
circuit = hhl_circuit(matrix_entries, rhs_amplitudes)
state = Statevector.from_instruction(circuit)
assert circuit.num_qubits == 5 and circuit.num_clbits == 0
print("Register roles:", [(register.name, len(register)) for register in circuit.qregs])
print("Exact eigenvalues:", np.linalg.eigvalsh(A))
figure = circuit.draw("mpl", fold=30)
display(figure)
plt.close(figure)

## Core, minutes 5-12: predict success, then inspect both branches

For the default $b=(1,0)$, solve $A_0x=b$ on paper. Predict the **sign** of the second component and the probability that the flag is 1. Computational-basis probabilities cannot distinguish $(3,-1)$ from $(3,1)$.

The success branch is the unnormalized vector $x_{\rm raw}=A^{-1}b$; its squared norm is the probability of success because the rotation constant is fixed at $C=1$. Only after checking both raw amplitudes and probability do we normalize. The failure branch is $Fb$, where $F=\sqrt{I-A^{-2}}$ is the positive Hermitian square root. The cell below computes these as **classical validation references**, not as circuit preparation instructions.

Exact three-bit eigenphases let inverse QPE clean count to `000` on **both** flag branches. The flag is retained for postselection. This statement does not extend to nonexact phases: a label-dependent rotation can prevent complete counting-register cleanup.

In [ ]:
# reshape axes are success, system, count; do not swap the flag and system axes.
joint = state.data.reshape(2, 2, 8)
raw_success = joint[1, :, 0]
raw_failure = joint[0, :, 0]
classical_raw = np.linalg.solve(A, b)
expected_success = float(np.vdot(classical_raw, classical_raw).real)
observed_success = float(np.sum(np.abs(joint[1])**2))
np.testing.assert_allclose(raw_success, classical_raw, rtol=0, atol=ATOL)
np.testing.assert_allclose(observed_success, expected_success, rtol=0, atol=ATOL)
np.testing.assert_allclose(np.vdot(state.data, state.data), 1, rtol=0, atol=ATOL)

# This integer-spectrum reference avoids sqrt of a tiny negative roundoff at 1.
eigenvalues, eigenvectors = np.linalg.eigh(A)
known_integers = np.rint(eigenvalues)
np.testing.assert_allclose(eigenvalues, known_integers, rtol=0, atol=1e-12)
assert np.all((known_integers >= 1) & (known_integers <= 7))
failure_reference = eigenvectors @ (np.sqrt(1 - 1 / known_integers**2) * (eigenvectors.conj().T @ b))
expected_joint = np.zeros((2, 2, 8), dtype=complex)
expected_joint[0, :, 0] = failure_reference
expected_joint[1, :, 0] = classical_raw
np.testing.assert_allclose(joint, expected_joint, rtol=0, atol=ATOL)
count_leakage = np.array([np.linalg.norm(joint[flag, :, 1:]) for flag in (0, 1)])
assert np.all(count_leakage <= 1e-9)

# Normalize only the successful branch, after checking its raw scale.
conditioned_system = raw_success / np.sqrt(observed_success)
classical_conditioned = classical_raw / np.sqrt(expected_success)
np.testing.assert_allclose(conditioned_system, classical_conditioned, rtol=0, atol=1e-9)
np.testing.assert_allclose(np.vdot(conditioned_system, conditioned_system), 1, rtol=0, atol=ATOL)
expected_observables = {name: float(np.vdot(classical_conditioned, op @ classical_conditioned).real)
                        for name, op in PAULIS.items()}
print("Raw success amplitudes:", raw_success)
print(f"Success probability: {observed_success:.8f}; failure probability: {1-observed_success:.8f}")
print("Conditioned system amplitudes:", conditioned_system)
print("Count leakage norms [failure, success]:", count_leakage)
print("PASSED: complete joint state, raw scale, both count branches, and normalized solution")

In [ ]:
figure, panels = plt.subplots(1, 3, figsize=(11, 3.5))
positions = np.arange(2)
for panel, component, title in zip(panels[:2], (np.real, np.imag), ("Real amplitudes", "Imaginary amplitudes")):
    panel.bar(positions - 0.15, component(conditioned_system), width=0.3, label="HHL conditioned")
    panel.bar(positions + 0.15, component(classical_conditioned), width=0.3, alpha=0.6, label="Classical reference")
    panel.set_xticks(positions, ["|0>", "|1>"])
    panel.axhline(0, color="black", linewidth=0.6)
    panel.set(title=title, ylabel="Amplitude", ylim=(-1.05, 1.05))
    panel.legend(fontsize=7)
panels[2].bar(positions - 0.15, [1-observed_success, observed_success], width=0.3, label="HHL statevector")
panels[2].bar(positions + 0.15, [1-expected_success, expected_success], width=0.3, alpha=0.6, label="Classical reference")
panels[2].set_xticks(positions, ["Flag 0", "Flag 1"])
panels[2].set(title=f"Success probability = {expected_success:.4f}", ylabel="Probability", ylim=(0, 1.05))
panels[2].legend(fontsize=7)
figure.tight_layout()
display(figure)
plt.close(figure)

## Core, minutes 12-20: postselect real shots and detect the relative sign

For the default solution, predict conditional $\langle X\rangle$ and $\langle Z\rangle$. Why does measuring only Z miss the sign? Each basis below uses 4096 local V2 samples with seed 2026. We rotate the **system** for X or Y readout, and measure its bit and the flag **on the same shot**. We then retain system bits only where that shot's flag is 1. Independent marginal histograms cannot supply this correlation.

Each selected bit 0 contributes +1 and bit 1 contributes -1. Conditional error bounds use the number of **selected** shots, not 4096. Empty selection is reported explicitly; it cannot define a conditioned mean. All deterministic amplitude checks above remain primary evidence.

In [ ]:
shots = 4096
bases = ("X", "Y", "Z")
results = StatevectorSampler(seed=2026).run(
    [measurement_copy(circuit, basis) for basis in bases], shots=shots
).result()
sampling_summary = {}
for basis, result in zip(bases, results, strict=True):
    system_bits = result.data.system_readout.get_bitstrings()
    success_bits = result.data.success_readout.get_bitstrings()
    assert len(system_bits) == len(success_bits) == shots
    selected = selected_system_bits(system_bits, success_bits)
    selected_count = len(selected)
    success_frequency = selected_count / shots
    success_bound = 6 * np.sqrt(expected_success * max(0, 1-expected_success) / shots) + 1 / shots
    if selected_count == 0:
        print(f"{basis}: no successful shots; conditioned mean is undefined. Increase shots and retry.")
        sampling_summary[basis] = {"selected": 0, "mean": None, "success_frequency": 0.0}
        assert abs(success_frequency - expected_success) <= success_bound, (
            "No successful shots: conditioned mean is undefined and the success frequency failed its bound."
        )
        continue
    assert abs(success_frequency - expected_success) <= success_bound
    mean = float(np.mean(1 - 2 * selected))
    expected_mean = expected_observables[basis]
    # A mean of +/-1 has variance 1-mean**2; one count changes it by 2/n.
    conditional_bound = 6 * np.sqrt(max(0, 1-expected_mean**2) / selected_count) + 2 / selected_count
    assert abs(mean - expected_mean) <= conditional_bound
    sampling_summary[basis] = {"selected": selected_count, "mean": mean,
                               "success_frequency": success_frequency, "bound": conditional_bound}
    print(f"{basis}: {selected_count}/{shots} successful shots, conditional mean {mean:.5f}, reference {expected_mean:.5f}")
assert circuit.num_clbits == 0
figure, axes = plt.subplots(figsize=(7, 3.5))
axes.bar(bases, [sampling_summary[basis]["mean"] if sampling_summary[basis]["mean"] is not None else np.nan
                 for basis in bases], alpha=0.7, label="Same-shot postselected samples")
axes.plot(bases, [expected_observables[basis] for basis in bases], "ko", label="Independent expectation")
axes.set(ylabel="Conditioned expectation", ylim=(-1.15, 1.15), title="Relative phase tested through X, Y, Z")
axes.axhline(0, color="black", linewidth=0.6)
axes.legend(fontsize=8)
figure.tight_layout()
display(figure)
plt.close(figure)
print("PASSED: sampled success frequencies and every defined conditional mean meet the stated bounds")

## Optional expansion, 15 minutes: change the input, predict, rerun

Change **only** `matrix_entries` and `rhs_amplitudes` in the parameter cell, then rerun that cell and every subsequent code cell. Predict the result before each run; the answers here are reference checks after your prediction.

- Set $b=(0,1)$: does the negative component move? For the default A, the normalized answer is $(-1,3)/\sqrt{10}$ and success remains $5/8$.
- Try `rhs_amplitudes = [1 / np.sqrt(2), 1 / np.sqrt(2)]` and then `[1 / np.sqrt(2), -1 / np.sqrt(2)]`. These are eigenstates: success becomes $1/4$ and 1. The system must retain the prepared eigenstate, including on successful output. Never normalize a zero failure branch.
- Try `rhs_amplitudes = [1 / np.sqrt(2), 1j / np.sqrt(2)]`. Predict which observable exposes the relative complex phase. The answer is $(3-i,-1+3i)/\sqrt{20}$, with X/Y/Z expectations $-3/5,4/5,0$.
- Restore `rhs_amplitudes = [1, 0]` and set `matrix_entries = [[3, 1], [1, 3]]`. The normalized solution is unchanged but success falls to $5/32$. With C fixed at 1, absolute spectral scale matters, even when the condition number is unchanged.
- Set `matrix_entries = [[7, 0], [0, 7]]`: any normalized b is retained, while success is $1/49$. A global phase in U becomes observable under control; dropping it would break this example.

The source rejects unsupported spectra. It does not automatically rescale A, normalize b, or approximate nonexact eigenphases. Try the handled rejection below as a reference check.

In [ ]:
for unsupported in ([[1.5, 0], [0, 2]], [[1, 0], [0, 8]]):
    try:
        hhl_circuit(unsupported, [1, 0])
    except ValueError as error:
        print("PASSED: unsupported spectrum rejected:", error)
    else:
        raise AssertionError("An off-grid or aliased spectrum was accepted.")

## What this lesson establishes, and what it costs

This example uses **five qubits**, three exact counting bits, fixed evolution time $\pi/4$, and fixed reciprocal constant 1. The reusable source contains two QPE directions (six controlled-power blocks), two Fourier transforms, seven pattern-controlled rotations, and preparation of b. The reciprocal is a seven-entry compiled lookup. Dense 2x2 validation and exponentiation, input-state loading, gate synthesis, postselection, and measurement are real costs.

Counting powers as naive repeated U calls would give $2(1+2+4)=14$ before synthesis; this implementation forms dense powers and does not literally perform 14 black-box calls. It does not claim sparse-oracle scalability or quantum advantage. Reconstructing a classical solution vector would require additional measurements and scale information. Our classical solve is a test reference, not part of the HHL circuit.

Full source tests additionally cover every unordered eigenvalue pair in the supported set, complex eigenbases, all reciprocal-control labels and both flag inputs, invalid inputs, and input immutability. Exact-spectrum cleanup must not be generalized to finite-precision HHL. Shor remains a separate milestone.

References: [HHL original paper](https://arxiv.org/abs/0811.3171), [RY rotation convention](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.RYGate), and the local [HHL design contract](../HHL_PLAN.md). IBM's [PauliEvolutionGate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.PauliEvolutionGate) uses the opposite exponent sign, $\exp(-itA)$; this example deliberately uses the positive sign for direct positive eigenvalue decoding.